# CinderTrace — earlier Sri Lanka history: all batches, one download

Run this notebook in order, or use **Runtime → Run all**. It covers **2000–2021** in separate three-year cells, then downloads **one combined ZIP** at the end. There is no manual handoff between batches.

Your existing 2000–2002 run is reused. Keep `NEW_RUN=False` and the same Drive folder/active run. Completed years are checksum-checked and skipped; interrupted requests resume from cache. No benchmark or frontend files are changed.

The data helpers and pinned run contract below are copied unchanged from the executed notebook so existing cached years remain compatible. Only orchestration and download packaging are added.

Enter your existing FIRMS MAP_KEY at the hidden prompt. No Earthdata password is needed. Network/quota failures stop rather than becoming zeros; resolve the service issue and rerun the failed batch cell. All later batch cells are independently named and safe to rerun.

**Cells:** 2000–2002; 2003–2005; 2006–2008; 2009–2011; 2012–2014; 2015–2017; 2018–2020; 2021. The last cell packages all completed year ZIPs, their hashes and shared evidence into one download. It refuses to label an incomplete history as complete.

For a one-time download of only the work already finished, run setup/helper cells through the batch/download helper definition, then call `download_history_bundle(require_all=False)` in a new cell. This makes one clearly labelled partial ZIP without downloading any more data.

The original executed notebook and results remain intact. Source availability, NRT separation, unknown coverage, partial-month totals and the early Aqua rule are unchanged. No calibration or scientific validation scores are produced.


In [63]:
%pip -q install "duckdb>=1.5,<1.6" pandas pyarrow numpy shapely


## 1. Dedicated run folder and immutable run contract
This folder is separate from all prior checkpoints. Leave `NEW_RUN=False` when resuming. `BATCH_SIZE=3` limits the next processing invocation, not the full target history.


In [64]:
from pathlib import Path
from datetime import datetime,timezone
from io import BytesIO
from urllib.request import Request,urlopen
from urllib.error import HTTPError,URLError
from getpass import getpass
import base64,zlib,hashlib,json,sys,time,uuid,zipfile,shutil
import importlib.metadata as metadata
import pandas as pd
import numpy as np
import shapely
import duckdb
from IPython.display import display
assert tuple(map(int,duckdb.__version__.split('.')[:2]))==(1,5),'Restart the kernel after installing DuckDB 1.5.x, then resume from this cell.'
IN_COLAB='google.colab' in sys.modules
NEW_RUN=False
BATCH_SIZE=3
REQUEST_PAUSE_SECONDS=1.0
BENCHMARK_REFERENCE={'dataset_id': 'lk_observed_standard_benchmark_v1', 'observations': 38995, 'year_counts': {'2022': 8593, '2023': 13366, '2024': 17036}, 'source_checkpoints': {'2023': '509afdf0418a01b248671addb4d7a221a02e0e5e275fb203aa6e170206d29077', '2022_2024': '6203a50c88921de8c29757192fbcbd9ee2af3ee06bd7f923e5dcf50cc34deea5'}, 'boundary_sha256': '838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'}
HELPER_CODE_SHA256='d5514ffb85e732ee0c74de05de5328eac3d2073da85345c4715f36aabfa41f44'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/cindertrace_atlas/rebuild/lk_history_2000_2021')
else:
    BASE=Path.cwd()/'atlas_lk_history_2000_2021'
BASE.mkdir(parents=True,exist_ok=True)
active=BASE/'active_run.txt'
if NEW_RUN or not active.exists():active.write_text(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8])
RUN_ID=active.read_text().strip()
assert RUN_ID and '/' not in RUN_ID and '..' not in RUN_ID
OUT=BASE/RUN_ID;OUT.mkdir(exist_ok=True)
(OUT/'years').mkdir(exist_ok=True);(OUT/'snapshots').mkdir(exist_ok=True)
print('Run:',RUN_ID)
print('Results:',OUT)
print('Verified 2022–2024 benchmark is referenced only; no benchmark files will be written or re-downloaded.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run: 20261114T211825Z_97d5550a
Results: /content/drive/MyDrive/cindertrace_atlas/rebuild/lk_history_2000_2021/20261114T211825Z_97d5550a
Verified 2022–2024 benchmark is referenced only; no benchmark files will be written or re-downloaded.


## 2. Original polygon and tested data helpers
The geometry SHA256 must match the benchmark. Helpers retain native confidence and processing quality.


In [65]:
BOUNDARY_BYTES=zlib.decompress(base64.b64decode('eJyFfUnLtseV3l8xWpuPmk9Vtg1ZJRDIMnghupVGxG0ZRb0wxv8911VVZ6jntfJJBttvnbvGM0/P33/47W9//emH//KHH/7rTz/+9p+//vQvv/z5zz/9628///KXH/74hx/+9/nj/wXA//r7JygB/vrrL3/96dffft4gf//h53/DfyX8/S8//scG/Z+//vyH//bjX/7Pjz/8A3/9959++Y+ffvv1bxtWZ/vv//nn337+H7/8+W//ftb8119++fXffv7Lj7+dZfGPrG8ibeT+xz/Mb2WsPtr40x//cP5eS8tj7X9kj8tKIjY+eyurn/Fxvpc5+kr7n2ZgtY71TDPKqsvGvyz/J47src2c2+r5TkiYWmUWKXfRdeeYqZXZ6wHA7PMuN6YC5DLHrOev68DVJHmcqWuxicaYsx2wueFKa2nlu/1q87WW61mwpDxwM+cTWzDNVVKKE2XBOW04rZRzuLva0pB2764a3HcvwC9LpmBPPSzZsT5e6XydxS58zTI3QFu1dtuGAczSxkzhxdpMXbr+ZRrcSLmPsLmGGx1JF9TLElxfnhFR2tiPc+BKt/kWLqXH+UYd2fBtcep6hjfCNCBg7rr97G9Yi556P1Frc81678xOuVYvo4braphd6n18CcvOKfkAABnKPAB9GUBJ5VxTX1WK3OM7AUifussDN4Ykvbji9LY6bj5e0xQpza4H/0fGedc8McWF9NfoLcXb45oz1y+v26WusgH4+vPeaCm+3+/gUcA5cAQgkL3J+pbxgnUqHuuTjIIZ7xVssCSldyO7VUYGSd2N7PFcCjjJ3fsIcKNNI2PAgTaBxR/kCXQrRu5y5kvkDB9XvnIF81nPumO1qU83hRi3/w7MGO0Ts8FnZMwcv2+VN/7JVkapeLG47yapGvvCZc/a9t/xOMZDes357H7gjpfej+1ugIXVai/OWVstQ59ODA5PqssC5bNyD6MYcARc0HNdo4Dt6TUBmVvfqEe8L236e/Q0I2qDtfRqfO+BKy2Q5JIOLLb3z9hWlTiP9CZGq3OBqXdF+D0O9LdtzAXKLCWgPuBLXpcS7RVwYbP3wEAWtpWV/4nd60qjP/xv9d6X4mfzV824pnyupZFYPylpNUhQFQ3n/nGQpnBGupCHpUlED2CBMZAxIRtyj9PIbEU+8YFSti0XFoADbuZxlzOhMiaouRx0xanAdu5E1SYCw6uK7IeqifW6oG4bqN56PePgBFXJ1Rj/97iDMxIgLbDZZcT61sEgl7IhQ9UhYAz1jNc+h/154WV7ODp4YSnDr6bgyiKhg7YgcKfxNH2JKq0984Cfgx1euOlHH7k44i+IUogiUWGYfT5OEJ6uycDSyohMhqTWZy5h3QahYeoCODvQ/xw7UR1QZLDzgQy7is5zvjZakytNuhjc9645Kl/4d8Yzgn9S6nySQSbtRCY4ypbAd07jclCUDjZRbeD9n3l0GKwYVB+PsKBd6GPYVc06Zsln7wtKlr0dBdRSJjvPeMej3zfJNkEDV53y+xtpeTV7oQPWR21fyL8DsMsD10AHPg11yXh9YCvNVUgIYQjB+HmmcJtKHDoNFVo9wwFbDVgVRMs4RIj5IZBMdYTSQRkTnhoLLuXChsrgYuCKe1xAq2HdVGe8hT7Ak1VbaK5tgRTKiHBC3v1FA4fm1uv7OvhC9PLt9hNwToXawXhwfyiC9y8GB/It+Z0PInYoBdp0UF5UHq375hVqmk/T0vtKEG9gf/cLA4P6kmbYFc48m3Ic01G/TzNBdwK6QzE+QOAdKt+rK+q9UVfzI0JLgnWibNbtNEj45yZmIy7dvdl0wOGpNwr+ruq3cWvqVuNZDgiVDQdMn/39fUdjpEiWhzimQHVWWRS0VqgJqpBsuAFTYxkDFYrWEi+091YVbUzDEuJu7w9cwlXpy9re8bj9QYeOW2lf+LaAM+TnHQUCURHFJdx3jxn5aSP+BljJpAG9PsdsGEs5UsCAJmhyNFCeQMCpZNpwa2vqbqN+Z727t5m+FdjH581n7WAc+3n4915lSA33NSHRoNNEJkI4IH3rexN4PNy76t5yAWBul1XuJmElKlLpBLDzGgTvQQDIwnTvfvYLQMYuRzIW8lHF7mrjNY0UEQB0AmFpwx28RsJlNfCcbtrzBYOohlEYwcCymhQdBrrLcIsXw7iMqarb0NMKOFM/GhawJi9TwRyg0oQLb4xrB+dZkZ9w21B3RouaOYQX5KhqMnZ6bDyNMw6hONQm1usVPE6SZ+OQRKk9spFwkKegtrivDBU2qWai1wm2CdCokkKFvZohh1sLh9nDUA6x/sc9QUeAUIqsK8Oe6rpctflkQj2I24KIEODRHYeAqHm5sKIdsqozYp0HCAG16oyDyS919Ng4/Rm9hWtKDdtRvCmKJvTtgIGer/u15+pIPg7lsKjdiJcJHPscB/gRj32EqN7nMrgBQnGzCHAgyG6ce9i2IV3btOumPQL1Qufza8p8c/cwbDMb1qB+qa8LJCi9jLguUGAomk8DaxX25DHDIVrUDB+2r0xfWvSmLU5dFTttHoHZqqrqAYOoNN6xoEjBijpmIVQadbaJ7xfo3a5deRU0/hkcAZrc+TPMW5Vk2Q8A2irXnqzAT984LKwcfUOYF1JTtTWbAHZZG9eUnBSGNoCTqmJ5dtChDXZFbJ+gg3U+livesYYXy9xYHKceojfdbR5gowSZAThMAh3OTopLE7kDQKHlVzcpb8NJC70eqiXajVRoyqXFjbzzVPA81Zp0nukvBAMdp4/DUAcl2f4bmLNaIeelgeVVLSwHA9n16NrD+/alXNzY2AKVlB69r7g2yCXVGuyZBj69ALCwhvoIpwHgI+gMETVhvxXlyCadFpXBNeLGJqYeyhBIFCC1550zjH7VH88BM+io0Fv6vGOHjqESKRucgPL0PXs5tj0HcBVzYzR51HFp8M8wJ1d3ix7Dsl/zkROAg0kc7UjAdfKMx6dJuJlKzgcAn0Ai36MZAOinK0CBya6P5jP0ebyZE2LbnBlVh3nf6tLZF4FFoJso6SschGHveYSDyeJ21SDR5TLNmeP13ArNfJzeGwCGZYqUBjipakHbTWItKmjGGydlK/0sdz59CAgyPP8ep8te59GLxrbzTBFhaF3V1/tMuAJV5/GMQWEthi+42CIPotPFNdUNfMmTcAPiet/PAH1n5bZ6ehAa1D4VUwcsQWd79bwDN8528fy12t9h7+fHZYZN1KXoClRrvccIQJdRZNj2GhUBFWvn8wJVS13TBgeZnkqPy8h2w9g84LG5xHFMa+p9tVttDQjwwDXY2XYbhW7ViHzgJ1eJ2GdN4E7nK3pp5pfLzrPm5zTQbZM5EGye2o/s6AP6ksbNbJlCK9993TwrnshIkNQ9Dsr3gxP3euM2skStgB4lGDb+lis9ISrYjdnEnl0Grj+bJ/4sh3ltFrysccj7tB1y4bE+NxzmOcNL+SKoBOy1j2ePQv+0Km5GTZCmZY3nMsBfVKOtDgeJWZ4gyDii++NM2Ex7pgPvz2piFZ8OJ5PxTDdgZ+pxjXlARdkKsCMMyKypwOsBbpq/+Vz5fjtVBZ07wNIv+vWBA8YaoWefD2z8mY/GsjmYa3jDYrzlwOGWTKHz80IjH2ciOkkMlXDksaL0B0Fg375fcJRx3g/LFLU/Zg0AN+aIS8Flq4izFTKsjLOzSp1HdXl/sU9+SZlRplEDDNjUosKFyyrJ8QzX1KKeRYepPQv+57zYSe6hTq5aDAAor2ZRv8ckxRmzXwxengGIO3Na2voCND/6FxSToYobxYmA0SifOxPMudT8HbYD4Fw556YpV5+oHsfxl5ZjJKbnzZA/N4KLhsYT4SiJ2xMZ23AQzzWHEzdalWrJOMbACsrLnRAMWjLG0j8xH3RU78VL8WWgJhwFnN6WKvPzM2jiFx8h52cgTGiCEh+EamxzdKM5HW8V6FuaPIGOzSLpvnlCxYNHtHFG81p75gEqmaAzOOjCs9Rwq2CKwGcfBjXfGHPLTrEuHxlcPOMMELk8I5L1u7Eqys2bCQkIDvCU+2UOO4Ld1VtUOI9+qi8vPgHIbUU4+pfVoeuCHypMWZGDQGKBvpR1m2Sl97a5A4UPA5bZo499g4GjXGygc/3+GUIXBPhQeVrQipSuFYw26w1xYxNdydWGIb6nEhtVXx1gwF+iMgWkpgPy41YZe5PnUqrQTam7B2uEtIhSpjJWoVLGuB4ouZ/UFlyJO45MhDMSdnNIoOs01Y9sle7BiQOWGQXV0yocwxm9Rd0ekgO6tU6np2/4syt+Gw7qvqg/0IgHkgEM0H2a8xvNeOelUEvAv+ITg7vRtWjjkGdyPsTjygof5pQilYCrpTU+FB+gNG4jGt+QHTiNeiEVDOjdH6dHzjUb5hqraBDh49AQg+jVPKK2HtNKjlKLcdyvIki2laB4Pcot+Wr1cSlMe9oDWN/+vKghH/hMlVDVa304RiYmNUChfSjAOR3IEDchArXHYYqoF1F3TuzoK0e4ycCb4oxeVeeNJvc3cT5a0vlzPoj0VB2vOd+sxutPeomrekKvSB1G2w4Hcyck6gjtdpmmpEFEgqmNuAyVL305Q8ReGA9zNxThGCRUxPbrgs6hLtaz3r5u1W5sPuoI+Ww8M5/jSRvZAAJ6dXFAuCbGAzE21v07PWTT9wnidkyUne/Qm40DUUsucdqyZes9R7Prh57RSnwmnLc+8Z4NhqP289ygtfrhfc7UZbcScUTp3c8y7chMcKA1Xk8BYK8phthKk4aQcwzCOZfEZRRo6OeaZmBIWH9Ic/tHvk1oHBYcz3bdgyFUd/oI8208ryEbvkCkpBbRd9LkttimEhz9CCunOB9lsu534PmZTufoSxXbDguxN6dqRmcVCLNmiVC2CgN/4xwL7NR88dnW4WOPOBHk9nKxptc/yPJTDWgDO6pN86or2BRq4oFahAFhFQMmTCCgsMkzDySipRuZPShUEHokEwhmPJSqSfosQpeTeK6E0GSTbuaNPgsuF2ats3MBg6eWpedUMIhiaDcB+6GJq8sdw6NPM7z2MDMuW9IDZ4MDIT9wBcRazJvdDA70MHO4/dLpyfnwL03qZ3UjSWYOoTknfF8QzuqZ3bcFZYce5XsvijYTbztSXA+SpYz1ZF7cc1Zg3QBP7dBolBh8PQFTcuIHnNAD8PmKQH1InrwBoBvnkT8nYqw8uCLHN0azu0ZHDP0gKcoO4wKg0/2jN2QTVQYa1Kuw4Zjzqml8ZqxOZtqISyvAVcgF1crMA0Bqa00iHLhwM1+avfRW+hcBZHJJFdg+EXEshxNCL14mzmz/oDjPV9hg4K4BfzFjC7HNQU8dxKmdizRRzoeJiZl6g0ZQoMFy3hTSe/TxaRxBK+IURiGD+TetqxPTWArUJ15YePvRNvrdLxWMtkl1BgawROfpk2244RiB9ujb+AbVHzLZxin7Zb8AbCZg/5PRSgBozec7ELS7erA4pPS+EIhleqx0YGRw/B4WbJRjRpvGymCUjDPOxAYTkANSJcuZuIL/WZxWqZXBYXMy7YtsUK2SeQmchacRrHyAZSqOKihUnlBWdg7XnQM83muGod5ktjAL7AGQvhrXJtaZnFw9J2TQSigynlyxDUfRciait8bWc7UDrLs9C3YqGHoLtjHgSKs9LtiY9mb6JkO6Ei4JuuBytZ4BibniKoV+Id2ugVF4uU4JMBiyphUa32sweEfwPI9veHCaTRfOloWEkpD3A7jOJPsn857WjtAQiNyGpulUvmJyAHpbsRyGvT/GFpa65IwrVW5mrQgHwKwhBVMuaiVWbmyAGtdH/fQ5g/3sHD5HbjrBy5ubvCeCUGtnvBXD7co85/MZuFHW8xgXAocBc9mvSWeoiX8zWumozH0dgM7U7I+FsTcIhDPDhLFrHMK9E/3ovYOOqpNAuI/oUQR686PsAdlkK1+wlfAc5CQBAcE7gPIWjIFC1w9jyY0KcPp4FOhxtUtEGog8xsU+FwJqzf0mULNhZLiXi0k+kQIYv87ujSL5dQ9jYBxUazqmu5XSKOXhpBmakN0HraoRKl+4DMP7iosWw1qsd3CNasNB+ClPsOPg9ms+EzHffvn3VGvP38kAVDD4PhsD/vG8UGLFbFybh9bjjPvNTBtUajIw3ESKpJ13lYqqPhYEBBpa/OKuWsGGLDbXt056vrq7ZzKeXUtiRum5j0QvpEWhGcd8eEFirNninQrWGHkaYZ+Jl2reOwvOgnRH8PIRDvj18doHbu5XTkKj3NMreCy3QzHeGdlQS9+yVCB3LQa3j4VnwRE1WKjBehBfkpDuQLjeLAfNsgsm6NVqBDZrSBAM1QoxFE5ojYT8P8JBfzNXj50DLLaGoO2gCir6XmnX1cy08QJSqS5L9gGrL+93eMZqeo6eS+hbxeedaSVgZ7a/trOO7f44npjPc7+3pCoiXRsRDpaWJ5jbfEDZltcDB5ZiOVPAgRb8Tp0acsoWxLAkLXDMuc44azn871U/m7yg+8qe2wXpyeSgznSibo6q4d8zo8OvqzM7gCbXhdNcACiLacwV9zloq3wmiYEO+k552xMVN02LAUh1AXgmgjSy7yfkwhjPfpiFp0qRPh/lm4Q8ZcJBwbFpoL634TFVDEPwLmMemntBbSeWAxIOMrsrn7BUtMKqKjd/CQdbQIsDLLlpZCZKDmOenXTKNMJLZ7Yug3X1wR7IVMv760zcaCVOAwQ02mXZxkpnu0CtrJVzosjZmSgVrJcNBymidu0yOIZjDwCljLIx2ydIBAShvouzT6bTKnZTMq4QKeG9YG+2jwZ5RCnOv0PPtlQ7sMcxxpmPB33tvcQaPphy98NEtfiDHhp0/P6SKcMVVuWoG4RKDqujXAwYy2o2lwHU2UIdyj5BdgpqzMtd6RkfPVs1lNg8jLm442Bj0rQ4es6+3r6iAMe6AkuT0xtq1B0vIUHBnuszP45OWTCxA5CqJTnbOEbrcr1mU8gwF6blxYHjjxZy07khchXVMnXjlTh39kNHY9ODl0Gzx0P6m6F2Mc6guMDgjYRwEhkzs3e6jVcY4kdypE5d6YNWC3Oa2+X4ZCI6l81QWWsYNZzEvHXLFytAmRbCfxQpsMvNCV5snrlt0CBCgItDE8E8j7gsqql2c0eUTUUWQzFwWGhQ0RJOzPscnyjEjKKR1jMfNtJtX5mOhajCkIe0bN+nxeDB+RA8Qr/LfK+R4r0wz0pHcY8gwf3nsXVmHSjMmT4PsklKZUIzAOBEu5rm5vR3a3pBiQQbEsSoGWemeNyLXAbHONxRumDRddU+i75smniO52VZkrsso8IWZEXdOFrgokGmKDINoNL8P5YLH+V1fwCgZ/D6xzDiG5iEVjCm0jQJT0rvWJL8uV4VGdcQY7WW0sk0gAItcB2LlLm9w//ekgXM9zgwtyj9mKBj9dE5T93hEHUg2fw4WhsRl9tOx1PjxIoVF8vxrndhWlnRIh8eOZyzFfArK3bTuhyInk67+7zrhuu7fup+2Qyutm4pqxuO1aj65bRa18n02ePtgAQS+SxFZwJLLdGyZiy9lSfquSfKvYorAoMx+b6soGkxHel5SBBUnf3JDdzlk7Ax8z4Zo4LNNGcrBcIfLHVnn4xRSFEmZUV7rG9N54p2RbqJAi9xgr1cSzgZVPrh9bMtYfHoV+ET5fRY+LswrVfL8T5wEHPdjmr7KYW+obBvVn3W6sMyN94w1tGm1WlgSS/D3uN0ShnTttMURrfanQCbfEj7ALSZ8rNPkqjiiFfIrVRydJFgn90iaV4XTDzOz/1Al/JqHCtqaqQC1+j2upCDqql7yRj1H4/n8BzgEV7gtzhT4MnQ/sTj2cWnoZ8hbr8zpXF8WW4wuh+31Zle9QWN2o57n/enWejnAsrJvSCmekVH4Bn39+7newGnzI5+qT7u3UXf/Hg0w1PHXSxKeg++HcOG57DOVuQcTCde68lWPtXoTSeAKdTlMWHOSWEOOcsQugtaU2nQHZHw6sMNC2EejniykHefWNADPJQgTDrEH2Ii46Ef6d2TcQgGglXL0qvlIfKtamIcuMJWBoGAZP+d+ZPDXKt2flx2FQnbIUeQ/AT0N9wke7f7JBwJ7kmIOHBz5hhvhcyFCaKOFOcjvFOnO4aMAvdgvkKK0U4K+Kn34S0/oCBI3nEyekXEKzg7jLp5zp2GWOGF3wtQdrz73AWpT97phuuDFe1hn9ioFWdWn4+p1Z79uOFgtKn9afhNb/Y5EZSdUp/0gz3ei5gYMXRrX+EoBM8BoTLYhQhQM5QecBTiJPReSLtekX8XupA/jzt679O1ccDtsJtei188DUK3x4V+VGbj3pWdemYq80GvtW5Sxqk7bqbYiU1j9m9gc3TrBXxgvdpS7PMOMQXvlSNWgbhmsexlpxoYNBLDrFBtZgrMuZsT68ZFGVLzbh6MxsVki06vff9smSGsAy41bJv566VHbW63NBAWEYflgBm4du+WQq1//70wRGQF80yqXGdecknVzZuvzxTpHo4pOxlDbTBbgE4nZZYn+YDkY2zBFgQHKMWTASbdlizKufPZ/SQJStVJ7gEOW/8YA8s0s04WEcNY6+NecMU3yQj4b6di34Eqp7vR7jfgy5aVdIANZKzLS8J5qvszOE7UG+9+mELa3mSuTGVVqd/2lSszh8PxWKbT+xNhO8vi25h5BzjqszpeWE93Bph42+rH+Wmm7ZyvAkbrpS5WNAx9QU69Bz/PFkXwKuac2tvpiIWI3nFByCtiCUEdvBcvAB7QGuK9EcnknzQZGni/vdMKBUjMu+5tdticKvYqocwJTa9wnzEjlLHurEqMo7OweO8cA6ZKi5FKDrP4ZT27BQcxp7B3EIE8GE9+fGUFlDlZbNOsyZWY/su7S/3JQ95w0MZPjiC0fe9MIb6vYU0JzmtDlR2al2siSfhST23B9m1MX2eU93wYL5aS5cuBM8q7XHEPj7WOYOcsyz1eB44pimrieWk6lNLgYOGy07Olndl1lqu4igk4eqDa/NheZcTbBdxk7LKs+an4MtuNKTOByHDY9Dq493ywsE6eOU7JrEmfIOdDHJVV3dmfv57s3gJT2zpmhQ4BfJrbVC03MzYdfRjQOBvKzMqz74CzJda/FNZlzifF59wnFNHLDgZz4T4uiDIixNs4D4wpK1B1dO+r9BkviCU4Fr+3i2SidI+pn4UuW0sGcm4CJnl6cmXWtEjgMqvFMkLYmTjnJ9rRLdk9oDPpIxHzF/kqfIuY5ps7pWnM0z1YwmSHw/vp2PjoJ8JUvxz0D4JBU1EW6ZcEwX0q5lgrsfLjWN/zMF0rNkFiMDBZIMP3U2BNxxT/xEfUbQVspFK1981zLlWIulNn79G8nvQJkUDvxo062eMoxeqQxCIHExGGzY3Z3CVcBDBtNB+vYHXFnXbMm6XQeXLR7/OtsU4eK8NVX26q03vqbIB5rEBlWweSs8j5O5j9NAPaFF+QqVUZ30TXwXyHjxeDwTKkRlVvybZpP/gDtDr8xzMzhH5jL8OwC59AdAu8HDjqrkbV3mUEsi/FDENWBWdfDSLibGew2kjNWrez6CyNKiO9njkFfwjVqbgNcjMV2cFvAnagTO6oYrTTl69D5uB+lZMv7LZlbTkaaWsXaukibqTRU/IkX7PNj9VvGhgLxx4czM1j/MHGBE5f3MPy3o+xsAzm4HgG+JNWuPXXmi0FvR849vNRXuBqLDB8ujOStMWEL1efl6aKC7DCkpWM4y7yWfcxbeKF+Pv0Dk4WsFlm8IVrlmUcGtIw9hbLWjMR2FpcuEsAJtKIIoxMyLOD3LrpTJePcHXrjjbemMR8mCXt3dj9yfpIHC43KCueivi94ZliVizhSCiKxO1354MSDYvVDk76uAonVppvR5JjkWRVzUtm7vW9JBtnvDgaEK/MZm5kaRExIfvA/hUH/aUa5eYBAN1qCNXdEDu/yO3OowRYAnVgW7TaYm02K0jdS5Hp7Xz2S/+G/sURAps5Xw/22flCRzIsN//ot4xFWva6+1hA3Le2Ba9opY5+PRWY3Ny6Z+dS6qifhDV5pBqFAuNzosV2RhiTTD8/indnMOSLs42pEO1URDHoMoI3j6lce4CC3b0csJDSU9nU2WTg08k1WOS5h4Ve0idZ47grVr6lgdRC9T58HFbdLQZO7hxzpwz74D4tTDDPsGJRlxEM0+WoBUCl97CrYdXWH8qtHWWBoxKQ92RkgfZTSw3+li2hW8F2e6xYzAwtJFmEz/p7rUZNK2IndAWqXvdlHW6r2iY0WNCem2VxWFdVSL76ViGxx1+TJx1/w+EN9jzQnlhqb39fcsiAOZfN12cFSYhaTSYPd9NAFx/mKf2aTEHrNl5YHBetw8lE1vVh30FaNKvWO10QsHnX+Xw6gS1y+jik4n1JyWVPH5sE0rNQffbP8mpxehByk6/9UmvRviiZRRlPasg+rexs6zBPYWutz9Zhi3hYIq+jN6BZva3vK10nydr1/rqf5ddyq2CpMhf1FBqRr12i6QlihJPiOZkKNhgG8QIXdl9hv5PP3nSLfuDH7FjMjW6fmvdOW8rP8UBwxToAWKQRnDB1d36x743QHrfxTlXvNtbp3eLxiRVAc2pn1a6ml6X6QP6NHOLehFt52cRt9hIygdgIaDGJ6R7EQpwky3I7hNLPEol0x3RZBt7iMtCvsxXbVZsHl9luw042lPgyPoo8Tf/oBHQTOES1oYI6sQOur2nVLNnC47vtdmw9x9YFloI5s8H1kZ+2WdRuWxo2zvSK3W+K0tkSPKgpgBbOB1It49BS6Cjf2nC1bbGgvMrbynpPNKAUuHq2aOkv61PjmQLMehk1bBRMBPxcZb8lDiRIzKfLIh0bVoZkaUyZBkswjdhvi3anBdknSTk2pKoM3HRDIGHXjxTO9zE+6G17+p5hG17RoGB84hLvHwo207k/js/02jJPWzP6gfPHcRLrVWvs+lfZVSe/FWqEY4p9icdi5aBlgurtsP1TimjdIIPFusHZK49L4Gu34HblwQCgrMnpDkfH6XqclHuc+c/xuUCYpHnbiKRS4zOxXVfL4ZlW6acZJ72TzS+WeBnbHHambAx/nzza7RHarNA43DflxW2aCsybhl47b/f0fGRnStWrHZ9ZXdPiuqyqUvewJfUxACTzdvdkxtz9ws5dqNKFd2K/5DmiJ2Tnz6QO5em0p63DUzsyzeaQybvb9678Wik7PWd3iYnbZcjZWLfD4RRPJ+ReGdZWdULfo7DRRop8FWzWayctNwWKO3MWzkTsrWkTkE3N2wCYnQ3Uh2IJTXQHjBpWaEtKsYfH/2K/jNPZl6lU8l4ZZKSXMej3LOZ8CZNW7agHc3extkpwS8AS5lfcxsK4AHXd2gSgvznkdig2QgUm1NNuGQSRPWnPtwfz0j5bydIrPUGMTWXcR7B2OuLyfTF2eje29U+/WZDVPTE5lTzRyQ1ABfMcubM81KZkp6MWpXRjTrg1EV0OB3y6EzAGb2l+aSfjnoHO1ssXbaxZZWKUJnLTxvVKdB9vMBDq05SZotc7eSsYW0yopXSuCLrjsHT4fzJNF/9xCXuLuttSRaHcdlf9t6CErTiBdKE+dZ+ylo9mS4Tb/XQjHJC5WcLr5hznRZmUrn9lQ4aHlBtdq5YORs9VCeFyfk4ZauN0l1RPJiBCZ6/qMs5FQZAuak6wAA3YWdIn2+bV2yearb/y4+XbAGlYOs2hbWhp3nDa0loTS5djF3c6j9t8ijAIxl/qiAdjVaWYe9auDSbEbevf6d02GV4nb+pwR4Ziv6BTp887dmWEYVezud+bYycbuMf9TubrvJVUZGpsSBhJc7AsdTyW6WbOzTsO3p65JKhPHTazzcc4PV4hzKelkhnAjvY8vXxZzKoTmTAYTBeMfWsH8wrNTHFpwuyDEvfFAgKLupjMbmVekQouWR6P3U4F5dSxNSqMezK7j12RgtSdeB5pUg1+63O5K/b8iLvaoQ+jN5OYc44WNQXQKujEDAImk6fIxJhDIOaucU185hwCf7xTEKNty+Am21rHVvyD3RwsIRZ6KLA3/mIG80GmOa1dsZ1lFHfbct+V2dM6Xln7Fn86AE8HDmfnAoNaUt5zL0e6YMkwQvTsB7zMpgGp9P6iEg2L8rHdxKK86V6EjXJ8x/sXt9wYdc9xW419dM0w5e9VzDMBblv8z9MTLvfonGlav1izQ+es5XY/ztMKycz7utjzc9VnnsZQc+AA2+6llHo6Q+9S9xIv79jHNZ9mzEl2x/+D3LbcYAglShxou1D4zW5mgH+4T46xqcQ2kDbO/jqeyLVjTtUzexYLo1sKx5HdvPHJuD37YN/bZx+jVas+HQ5Xamg8ss/FeOLdoft82mihcnc/Ry9mndl7tEKiicTAZpn9KSzYcEzVdWcP4ZrLanE3C9PC7/vuWmgbSM28JPVe9LJaGPfFMUHw6OnCLPDgH2KDj3auuOPTjwAvWAvTAM84+26puLYJ2K2y1Hh1nd0vlXkbmLA/7h4fOM74CPQuFmA/uoWwBsY9KLtVy/k77ZPPH1Fi9p3c+UEgYi2XDKXAC68eyKItS3w2/+jatkM8CElBvlSx4Av8u29se+utHMpIvQxPJd6PP3eEWNHdWEdr7vk4cJkZfG8vybST+erx7fC3C6yuxjgmGzH3qL4xxN3MZwOJDoy5GwHHM2vMdOwvvdm9p/xizOP60+pStSq7N3P/Uoyxyf1LNiD+4CSuRjDHT8jsFo3c+uuwzr0fB2HjN4/DaO+DP7gTMyUYV+3+Az+/t894FHbVvM5Dxrk/kwNYMnj7KX2cIhWJ4V66A/OuyznI4nDsLBBDCOywNcwq9MP87k7CdteOGQUeB7WuuavDywIgLreEKrO78yT4HvnjdZGytt5nGrShI5vk3nmGWBsIHx7sHhOnEbDI/OnAoWlzHGRCG2l82W0uuUaBRnep9zN0amw5zcdfRs+nNYirTrVzlKgUl83rjPExx1Tn2fuq/M2J5HK3rbQePxqr7yyry7f9ncfwnzrgrwSkQ6/VOunSAyNM5DVSoYdseaPc6VUvoN4RL5pZ+76UzzeylQgfuAam9jaB/Se7CRsFX6kzKqZUrCygZwuxU+hjFjSimfkwTAPMb+9RGmZMXjW+uI2k+BNJjR01l3/+/99N+KGLUTwr+4hXugOtNlLpeZApPHo+OxEaOXp+NB0BHh+lKlhlzEfz3nD8LQZVXZOl6ViE7XvbCifYPwL1aLk7dPG0y97BOGqT0R+6yyYtZrnyZns7XZMla3cl2/Fi2/TofmIZlnjIkr9rcP5OejO6dKEMU+Nsj73J1N3m4R3YzfleLY0q1aeH83WGV6KiNkZyrmFBR/qy1/G1QP2VkT+DhDjazNHoBzcSS+cMP8sn/dVfJxXFR0SfIFsp74sLlYmn6/3RKnah0V4uBr16uX+t09sG2KH5q3r9GIxCtd4GdrHXext4AjMofVlGHfSODtwW8PdLj4axnOAxsNk82JoReziLvWVegxfvbi4fr/RK09KN8jXYGU+9cM5UYZGNxyqprNBQgrDzMjWyPVYHMxQsl5st7MuL3YU/KxTbSpzoGSOGgQGxP9kyj4lH43bVWVyO/9fKRKIMSnKNOgZufKDxB1gOvrMDfWRdOfV35t6N87F9RPWQOTcIGWp5hi4c+QNazw/qMOvJRRVbl4TXZBLa+hpXhzAozy/N0Hs51Iloj8nO0i1Kxl2kYOUPUFsYaopnyli9PZs+tpGnK91fberDHNG+ff6wynx4DR1W7bPIDK+UW496NvPo6/hIamTyjHUNuLgDxtCjX/pq7ev8utigGuE6G2vZ8+P/4O/xqmHupixdhysqGvyRBu/07D9ROVk5fBaC0h+Uue37jUQIFLZgqmmodJh1lS7NEsD9JzKZJHTHqaV/LVZsVAPis7O7YsjvYKf+6E0d1Di8dgt6ptwfwBq7bdWZv/oGJP4EJuFoP/vPuLE++xIHtaqPLFXWmVery77MqNqvv/o9NLZpPfPMwl51dx5PsNmpGZHcSXxiN7WdsXKs9Jw9qZK/KDrdRODw/vWk++ImFL8re4OcZur6cRzwZw0sb8PWBPqsR8WAEBlWTBFMC4FydKxP+gusCMfk/KxFBddZDxcsMj7AaF/UHJejn8E6K3kBLPD3cRXjs+ktMrxwkPcQ5aXkXUx63yzkJpUR0uO5vcEfgbtX5pVQ0InPj2gxg9qaZrmGBG29nqdj/HE+8c9DTHNKpHrq4tWagwXF5nfeBf/84x9/+sf/A/jl+pU='))
BOUNDARY_SHA256='838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e'
assert hashlib.sha256(BOUNDARY_BYTES).hexdigest()==BOUNDARY_SHA256==BENCHMARK_REFERENCE['boundary_sha256']
REGION=shapely.geometry.shape(json.loads(BOUNDARY_BYTES)['features'][0]['geometry'])
BBOX=(79.4,5.8,82.0,10.0)
assert REGION.is_valid
x0,y0,x1,y1=REGION.bounds
assert BBOX[0]<=x0 and BBOX[1]<=y0 and BBOX[2]>=x1 and BBOX[3]>=y1


In [66]:
"""Compare native FIRMS standard-product API rows with the checked mirror extract.
Shared by local tests and the standalone notebook. Never logs or persists MAP_KEY.
"""
from io import BytesIO
from collections import Counter
import json
import numpy as np
import pandas as pd
import shapely

SOURCES={'MODIS_SP':{'T':'Terra','A':'Aqua','Terra':'Terra','Aqua':'Aqua'},
 'VIIRS_SNPP_SP':{'N':'Suomi-NPP','Suomi-NPP':'Suomi-NPP'},
 'VIIRS_NOAA20_SP':{'1':'NOAA-20','N20':'NOAA-20','NOAA-20':'NOAA-20'},
 'VIIRS_NOAA21_SP':{'N21':'NOAA-21','NOAA-21':'NOAA-21'}}


def parse_official_csv(raw,source,start,end,boundary):
    """Reject failed/error/changed-schema responses; clip a bounding-box result exactly."""
    if source not in SOURCES:raise ValueError('Unsupported standard-product source')
    try:frame=pd.read_csv(BytesIO(raw),dtype={'acq_time':'string','satellite':'string','confidence':'string','version':'string'})
    except Exception:raise ValueError('FIRMS response is not valid CSV; do not interpret it as zero') from None
    base={'latitude','longitude','acq_date','acq_time','satellite','instrument','confidence','version','type','frp','scan','track','daynight'}
    required=base|({'brightness','bright_t31'} if source=='MODIS_SP' else {'bright_ti4','bright_ti5'})
    if not required.issubset(frame):raise ValueError('FIRMS CSV schema is incomplete or changed; response is not an observed zero')
    # Whitespace is formatting only; platform aliases remain specific to each requested stream.
    codes=frame.satellite.astype('string').str.strip()
    satellite=codes.map(SOURCES[source])
    if satellite.isna().any():
        received={str(k):int(v) for k,v in codes.fillna('<missing>').value_counts(dropna=False).items()}
        unexpected=sorted(codes[satellite.isna()].fillna('<missing>').unique().tolist())
        raise ValueError(f'Unexpected satellite code for {source} [{start}, {end}): '
                         f'{json.dumps(unexpected)}. Received code counts: {json.dumps(received,sort_keys=True)}. '
                         'No rows were reclassified or discarded. Keep this source/date/code diagnostic for review.')
    times=frame.acq_time.astype('string').str.zfill(4)
    if not times.str.match(r'^\d{4}$').all():raise ValueError('Invalid acquisition time')
    try:
        acquired=pd.to_datetime(frame.acq_date.astype(str)+' '+times.str[:2]+':'+times.str[2:],utc=True,errors='raise')
        lon=pd.to_numeric(frame.longitude,errors='raise');lat=pd.to_numeric(frame.latitude,errors='raise')
    except Exception:raise ValueError('Invalid native date/time/coordinate fields') from None
    if not np.isfinite(lon).all() or not np.isfinite(lat).all() or not lon.between(-180,180).all() or not lat.between(-90,90).all():raise ValueError('Invalid coordinates')
    if not acquired.ge(pd.Timestamp(start,tz='UTC')).all() or not acquired.lt(pd.Timestamp(end,tz='UTC')).all():raise ValueError('API returned observations outside the requested window')
    frame['satellite']=satellite;frame['acq_datetime']=acquired;frame['audit_lon']=lon;frame['audit_lat']=lat
    if frame.version.str.contains('NRT',case=False,na=True).any():raise ValueError('NRT rows received for a standard-product query')
    if not frame.daynight.isin(['D','N']).all():raise ValueError('Invalid day/night value')
    if not frame.type.isin([0,1,2,3]).all():raise ValueError('Missing or unsupported standard-product type')
    if source=='MODIS_SP':
        numeric=pd.to_numeric(frame.confidence,errors='raise')
        if not (numeric.between(0,100)&numeric.eq(numeric.round())).all():raise ValueError('Invalid native MODIS confidence')
        frame['confidence']=numeric.map(lambda v:str(int(v)))
        frame['confidence_pct']=numeric
    else:
        if not frame.confidence.isin(['l','n','h']).all():raise ValueError('Invalid native VIIRS confidence')
        frame['confidence_pct']=np.nan
    for column in ['frp','scan','track']:
        numeric=pd.to_numeric(frame[column],errors='raise')
        if not np.isfinite(numeric).all() or numeric.lt(0).any():raise ValueError('Invalid native numeric field')
        frame[column]=numeric
    inside=shapely.covers(boundary,shapely.points(lon.to_numpy(),lat.to_numpy()))
    return frame.loc[inside].copy()


def native_keys(frame):
    """Published coordinates compared at 5 decimal degrees; multiplicities preserved."""
    times=pd.to_datetime(frame.acq_datetime,utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    return pd.Series([f'{s}|{t}|{float(x):.5f}|{float(y):.5f}' for s,t,x,y in zip(frame.satellite,times,frame.audit_lon,frame.audit_lat)],index=frame.index)


def compare_native(mirror,official):
    left=mirror.copy();right=official.copy();left['record_key']=native_keys(left);right['record_key']=native_keys(right)
    lc,rc=Counter(left.record_key),Counter(right.record_key)
    missing=[{'record_key':k,'rows':n} for k,n in (rc-lc).items()]
    extra=[{'record_key':k,'rows':n} for k,n in (lc-rc).items()]
    duplicate_keys=any(n>1 for n in lc.values()) or any(n>1 for n in rc.values())
    differences=[]
    if not duplicate_keys:
        l=left.set_index('record_key');r=right.set_index('record_key')
        columns=['confidence','type','daynight','instrument','version','frp','scan','track','brightness','bright_t31','bright_ti4','bright_ti5']
        common=l.index.intersection(r.index)
        numeric={'type','frp','scan','track','brightness','bright_t31','bright_ti4','bright_ti5'}
        for col in columns:
            if col not in l or col not in r:continue # sensor-inapplicable channels remain null / absent.
            a,b=l.loc[common,col],r.loc[common,col]
            if col in numeric:
                av=pd.to_numeric(a,errors='raise').to_numpy(dtype=float,na_value=np.nan);bv=pd.to_numeric(b,errors='raise').to_numpy(dtype=float,na_value=np.nan)
                same=np.isclose(av,bv,atol=1e-5,rtol=0,equal_nan=True)
            else:same=(a.astype('string').fillna('<NULL>')==b.astype('string').fillna('<NULL>')).to_numpy(dtype=bool)
            for key in common[~same]:differences.append({'record_key':key,'field':col,'mirror':str(l.loc[key,col]),'official':str(r.loc[key,col])})
    status='matched_native_records' if not (missing or extra or differences or duplicate_keys) else 'review_required'
    return {'status':status,'mirror_rows':len(left),'official_rows':len(right),'shared_rows':sum((lc&rc).values()),
      'official_rows_missing_in_mirror':sum((rc-lc).values()),'mirror_rows_not_in_official':sum((lc-rc).values()),
      'duplicate_keys_need_review':duplicate_keys,'native_attribute_differences':len(differences),
      'coordinate_comparison_decimals':5,'numeric_comparison_abs_tolerance':1e-5,
      'coverage_verified':False,'missing_in_mirror':missing,'extra_in_mirror':extra,'attribute_differences':differences}


In [67]:
from html.parser import HTMLParser

PLATFORM_SOURCE={'Terra':'MODIS_SP','Aqua':'MODIS_SP','Suomi-NPP':'VIIRS_SNPP_SP','NOAA-20':'VIIRS_NOAA20_SP','NOAA-21':'VIIRS_NOAA21_SP'}

SOURCE_SENSOR={'MODIS_SP':'MODIS','VIIRS_SNPP_SP':'VIIRS_SNPP','VIIRS_NOAA20_SP':'VIIRS_NOAA20','VIIRS_NOAA21_SP':'VIIRS_NOAA21'}

PUBLIC_SOURCES={
 'snpp':'https://modaps.modaps.eosdis.nasa.gov/services/production/outages_suomi_npp.html',
 'terra':'https://modaps.modaps.eosdis.nasa.gov/services/production/outages_terra.html',
 'notices':'https://firms2.modaps.eosdis.nasa.gov/content/notifications/firms/outages.html',
 'missing':'https://firms.modaps.eosdis.nasa.gov/api/missing_data/?action=fetch_data'}

class EvidenceRows(HTMLParser):
    def __init__(self):super().__init__();self.rows=[];self.row=None;self.cell=None
    def handle_starttag(self,tag,attrs):
        if tag=='tr':self.row=[]
        elif tag in ['td','th']:self.cell=[]
        elif tag=='br' and self.cell is not None:self.cell.append(' ')
    def handle_data(self,text):
        if self.cell is not None:self.cell.append(text)
    def handle_endtag(self,tag):
        if tag in ['td','th'] and self.cell is not None:
            if self.row is not None:self.row.append(' '.join(''.join(self.cell).split()))
            self.cell=None
        elif tag=='tr' and self.row is not None:self.rows.append(self.row);self.row=None

def evidence_rows(text):
    p=EvidenceRows();p.feed(text);return p.rows

def availability_table(raw):
    try:table=pd.read_csv(BytesIO(raw),dtype=str)
    except Exception:raise ValueError('Availability lookup did not return CSV; no dates may be treated as zero') from None
    if not {'data_id','min_date','max_date'}.issubset(table):raise ValueError('NASA availability schema changed')
    table=table[table.data_id.isin(set(PLATFORM_SOURCE.values()))].copy()
    if table.data_id.duplicated().any():raise ValueError('Duplicate advertised source date ranges')
    for c in ['min_date','max_date']:
        if not table[c].str.fullmatch(r'\d{4}-\d{2}-\d{2}',na=False).all():raise ValueError('Invalid advertised date')
        pd.to_datetime(table[c],errors='raise')
    if (table.min_date>table.max_date).any():raise ValueError('Reversed advertised date range')
    return table

def stack_native(parts):
    columns=list(dict.fromkeys(c for p in parts for c in p.columns))
    nonempty=[p for p in parts if len(p)]
    return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


In [68]:
"""Data-only historical rebuild helpers; authoritative standard streams, no calibration."""
from pathlib import Path
from io import BytesIO
import hashlib,json,re
import numpy as np
import pandas as pd
import shapely

HISTORY_YEARS=list(range(2000,2022))
HISTORY_SATELLITES=['Terra','Aqua','Suomi-NPP','NOAA-20']
HISTORY_SOURCES=['MODIS_SP','VIIRS_SNPP_SP','VIIRS_NOAA20_SP']
# A platform lower bound, not a claim of complete observing coverage on/after this day.
AQUA_FIRST_LIGHT='2002-07-04'
AQUA_DATE_SOURCE='https://modaps.modaps.eosdis.nasa.gov/services/about/'
HISTORY_POLICY={'policy_id':'lk_observed_standard_history_v1','target_years':HISTORY_YEARS,
 'canonical_source':'NASA FIRMS standard-product responses with raw checksums and native row provenance',
 'coordinate_reference':'OGC:CRS84',
 'nrt_rule':'audit-only; never append/coalesce with standard or use as missing-standard fallback',
 'unavailable_rule':'null count before supported platform/archive intervals or when source is not advertised',
 'zero_rule':'zero recorded detections requires a completed verified query; not physical fire absence',
 'aqua_lower_bound':AQUA_FIRST_LIGHT,'aqua_lower_bound_source':AQUA_DATE_SOURCE,
 'aqua_lower_bound_interpretation':'reported first light; local observing completeness remains unverified',
 'coverage_verified':False,'harmonization_applied':False,'production_export_enabled':False,'frontend_changed':False,
 'benchmark_years_excluded':[2022,2023,2024],'raw_annual_totals_comparable_across_fleets':False}

def digest(payload):return hashlib.sha256(payload).hexdigest()

def history_plan(available,years):
    if not years or len(set(years))!=len(years) or any(y not in HISTORY_YEARS for y in years):raise ValueError('Choose unique historical years 2000–2021; verified benchmark is out of scope')
    by=available.set_index('data_id');plans=[];windows=[]
    for year in years:
        for source in HISTORY_SOURCES:
            a=pd.Timestamp(year,1,1);b=pd.Timestamp(year+1,1,1)
            if source not in by.index:
                plans.append({'year':year,'source':source,'status':'standard_source_not_advertised','start':None,'end':None,'days':0});continue
            start=max(a,pd.Timestamp(by.loc[source,'min_date']));end=min(b,pd.Timestamp(by.loc[source,'max_date'])+pd.Timedelta(days=1))
            if start>=end:
                plans.append({'year':year,'source':source,'status':'outside_advertised_archive_range','start':None,'end':None,'days':0});continue
            plans.append({'year':year,'source':source,'status':'full_year_advertised' if start==a and end==b else 'partial_year_advertised','start':str(start.date()),'end':str(end.date()),'days':(end-start).days})
            while start<end:
                stop=min(start+pd.Timedelta(days=5),end);windows.append((year,source,str(start.date()),str(stop.date())));start=stop
    frame=pd.DataFrame(plans)
    for col in ['start','end']:frame[col]=frame[col].astype(object).where(frame[col].notna(),None)
    return frame,windows

def history_evidence(public,years):
    intervals=[];unparsed=[];missing=[]
    for label,satellite in [('snpp','Suomi-NPP'),('terra','Terra')]:
        for row in evidence_rows(public[label].decode()):
            if not row:continue
            found=re.findall(r'(20\d{2})-\d{3}',row[0])
            if not any(int(y) in years for y in found):continue
            try:
                if len(row)!=5:raise ValueError('unexpected table columns')
                doy=re.findall(r'20\d{2}-\d{3}',row[0])
                a=pd.to_datetime(doy[0],format='%Y-%j',utc=True)+pd.Timedelta(row[2]);b=pd.to_datetime(doy[-1],format='%Y-%j',utc=True)+pd.Timedelta(row[3])
                if b<=a:raise ValueError('non-positive interval; endpoint cannot be guessed')
                intervals.append({'satellite':satellite,'start_utc':a,'end_utc':b,'source_url':PUBLIC_SOURCES[label],
                  'source_year_day':row[0],'source_reason':row[4],'local_impact':'unverified','recovery_status':'not_stated_in_table'})
            except Exception as error:
                unparsed.append({'satellite':satellite,'source_url':PUBLIC_SOURCES[label],'raw_cells_json':json.dumps(row),'reason':str(error)})
    payload=json.loads(public['missing'])
    if payload.get('success') is not True or not isinstance(payload.get('html'),str):raise ValueError('Upstream missing-date lookup failed; do not infer no gaps')
    platforms={'Terra (MODIS)':'Terra','Aqua (MODIS)':'Aqua','Suomi NPP (VIIRS)':'Suomi-NPP','NOAA-20 (VIIRS)':'NOAA-20','NOAA-21 (VIIRS)':'NOAA-21'}
    for row in evidence_rows(payload['html']):
        if len(row)!=4:raise ValueError('Missing-date columns changed')
        if row[0][:4].isdigit() and int(row[0][:4]) in years:
            if row[1] not in platforms:raise ValueError('Unknown missing-date platform')
            pd.Timestamp(row[0]);missing.append({'date':row[0],'satellite':platforms[row[1]],'quality':'sp','source_url':PUBLIC_SOURCES['missing']})
    return (pd.DataFrame(intervals,columns=['satellite','start_utc','end_utc','source_url','source_year_day','source_reason','local_impact','recovery_status']),
        pd.DataFrame(missing,columns=['date','satellite','quality','source_url']),
        pd.DataFrame(unparsed,columns=['satellite','source_url','raw_cells_json','reason']))

def history_mirror_valid(frame,year,region):
    required={'acq_datetime','acq_date','satellite','sensor','instrument','quality','confidence','confidence_pct','type','version','frp','scan','track','daynight','geometry_wkb','audit_lon','audit_lat'}
    if not required<=set(frame):raise ValueError('Mirror schema changed')
    times=pd.to_datetime(frame.acq_datetime,utc=True)
    if not times.dt.year.eq(year).all() or not times.dt.date.eq(pd.to_datetime(frame.acq_date).dt.date).all():raise ValueError('Mirror UTC date mismatch')
    if 'year' in frame and not frame.year.eq(year).all():raise ValueError('Mirror partition year mismatch')
    if frame.duplicated().any() or not frame.satellite.isin(HISTORY_SATELLITES).all() or not frame.quality.isin(['sp','nrt']).all():raise ValueError('Mirror duplicates or unexpected platform/quality')
    if not frame.sensor.eq(frame.satellite.map({s:SOURCE_SENSOR[PLATFORM_SOURCE[s]] for s in HISTORY_SATELLITES})).all():raise ValueError('Mirror sensor identity mismatch')
    geometry=shapely.from_wkb(frame.geometry_wkb.map(bytes).to_numpy())
    if not shapely.is_valid(geometry).all() or not shapely.covers(region,geometry).all():raise ValueError('Invalid or out-of-bound mirror geometry')
    if not np.allclose(shapely.get_x(geometry),frame.audit_lon,atol=1e-9,rtol=0) or not np.allclose(shapely.get_y(geometry),frame.audit_lat,atol=1e-9,rtol=0):raise ValueError('Mirror WKB/coordinate mismatch')
    modis=frame.sensor.eq('MODIS')
    if not frame.loc[modis,'confidence_pct'].between(0,100).all() or not pd.to_numeric(frame.loc[modis,'confidence']).eq(frame.loc[modis,'confidence_pct']).all():raise ValueError('Native MODIS confidence invalid')
    if not frame.loc[~modis,'confidence'].isin(['l','n','h']).all() or frame.loc[~modis,'confidence_pct'].notna().any():raise ValueError('Native VIIRS confidence invalid')
    return frame

def history_canonical(parts):
    if not parts:return pd.DataFrame(columns=['acq_datetime','acq_date','satellite','sensor','instrument','quality','version','audit_lon','audit_lat','longitude','latitude','confidence','confidence_pct','type','frp','daynight','scan','track','geometry_wkb','observation_id'])
    p=stack_native(parts).copy();p['acq_datetime']=pd.to_datetime(p.acq_datetime,utc=True);p['acq_date']=p.acq_datetime.dt.strftime('%Y-%m-%d')
    p['longitude']=p.audit_lon.astype(float);p['latitude']=p.audit_lat.astype(float)
    p['quality']='sp';p['coordinate_reference']='OGC:CRS84';p['confidence_pct']=p.confidence_pct.astype('Int64');p['type']=p.type.astype('Int64');p['sensor']=p.source_stream.map(SOURCE_SENSOR);p['geometry_wkb']=shapely.to_wkb(shapely.points(p.longitude.to_numpy(),p.latitude.to_numpy()))
    p['observation_id']=pd.Series([digest(f'{source}|{sha}|{int(row)}'.encode()) for source,sha,row in zip(p.source_stream,p.source_response_sha256,p.source_row_ordinal)],index=p.index,dtype='string')
    if p.observation_id.duplicated().any() or native_keys(p).duplicated().any():raise ValueError('Repeated native identity; no rows automatically dropped')
    if (p.satellite.eq('Aqua')&p.acq_datetime.lt(pd.Timestamp(AQUA_FIRST_LIGHT,tz='UTC'))).any():raise ValueError('Aqua detections precede declared first-light support; retain inputs and review the temporal contract')
    return p

def history_grids(year,canonical,plans,gaps,missing):
    days=pd.date_range(f'{year}-01-01',f'{year}-12-31').strftime('%Y-%m-%d');daily=[];known=set(zip(missing.date,missing.satellite))
    p=canonical.copy();p['date']=pd.to_datetime(p.acq_datetime,utc=True).dt.strftime('%Y-%m-%d')
    for sat in HISTORY_SATELLITES:
        plan=plans[plans.year.eq(year)&plans.source.eq(PLATFORM_SOURCE[sat])].iloc[0]
        frame=pd.DataFrame({'year':year,'date':days,'satellite':sat,'source_stream':PLATFORM_SOURCE[sat]})
        frame['retrieval_state']=plan.status;frame['recorded_detections']=pd.Series(pd.NA,index=frame.index,dtype='Int64')
        queried=pd.Series(False,index=frame.index)
        if plan.days:
            queried=frame.date.ge(plan.start)&frame.date.lt(plan.end)
            frame.loc[~queried,'retrieval_state']='outside_advertised_archive_range'
        if sat=='Aqua':
            before=frame.date.lt(AQUA_FIRST_LIGHT);queried&=~before;frame.loc[before,'retrieval_state']='before_declared_platform_first_light'
        counts=p[p.satellite.eq(sat)].groupby('date').size()
        frame.loc[queried,'recorded_detections']=frame.loc[queried,'date'].map(counts).fillna(0).astype('Int64')
        frame.loc[queried,'retrieval_state']='query_completed_and_locally_checked'
        frame['upstream_lists_missing_standard_data']=[(d,sat) in known for d in frame.date]
        frame['documented_global_gap_on_date']=False
        for event in gaps[gaps.satellite.eq(sat)].itertuples():
            affected=pd.date_range(event.start_utc.floor('D'),(event.end_utc-pd.Timedelta(nanoseconds=1)).floor('D')).strftime('%Y-%m-%d')
            frame.loc[frame.date.isin(affected),'documented_global_gap_on_date']=True
        frame['observing_coverage']='unverified';frame['coverage_adjusted_activity']=np.nan;daily.append(frame)
    daily=pd.concat(daily,ignore_index=True);daily['month']=daily.date.str[:7]
    monthly=daily.groupby(['year','month','satellite'],sort=False).agg(detections_on_queried_dates=('recorded_detections',lambda x:x.sum(min_count=1)),
      days_in_month=('date','size'),supported_queried_days=('recorded_detections',lambda x:int(x.notna().sum())),
      upstream_listed_missing_days=('upstream_lists_missing_standard_data','sum'),dates_overlapping_global_gap=('documented_global_gap_on_date','sum')).reset_index()
    monthly['full_month_standard_detections']=monthly.detections_on_queried_dates.where(monthly.supported_queried_days.eq(monthly.days_in_month)).astype('Int64')
    monthly['period_state']=np.select([monthly.supported_queried_days.eq(0),monthly.supported_queried_days.eq(monthly.days_in_month)],['unavailable','complete_declared_query_month'],default='partial_archive_or_platform_month')
    monthly['observing_coverage']='unverified';monthly['quality']='sp'
    grids=[];p['month']=p.date.str[:7];p['native_confidence']=p.confidence.astype(str)
    for sat in HISTORY_SATELLITES:
        domain=[str(i) for i in range(101)] if sat in ['Terra','Aqua'] else ['l','n','h']
        subset=p[p.satellite.eq(sat)]
        if not subset.native_confidence.isin(domain).all() or not subset.type.isin([0,1,2,3]).all():raise ValueError('Unknown standard native filter domain')
        idx=pd.MultiIndex.from_product([[f'{year}-{i:02}' for i in range(1,13)],[sat],[0,1,2,3],domain],names=['month','satellite','type','native_confidence'])
        f=subset.groupby(idx.names).size().reindex(idx,fill_value=0).rename('detections_on_queried_dates').reset_index()
        support=monthly[monthly.satellite.eq(sat)].set_index('month')
        f['supported_queried_days']=f.month.map(support.supported_queried_days);f['period_state']=f.month.map(support.period_state)
        f['detections_on_queried_dates']=f.detections_on_queried_dates.where(f.supported_queried_days.gt(0)).astype('Int64')
        f['full_month_standard_detections']=f.detections_on_queried_dates.where(f.period_state.eq('complete_declared_query_month')).astype('Int64')
        f['quality']='sp';f['observing_coverage']='unverified';f['reference_model_support']='not_evaluated';grids.append(f)
    grid=pd.concat(grids,ignore_index=True)
    if not (daily.recorded_detections.sum()==monthly.detections_on_queried_dates.sum()==grid.detections_on_queried_dates.sum()==len(p)):raise ValueError('Canonical counts or supported-date ledger do not reconcile')
    return daily,monthly,grid

def verify_year_files(folder):
    folder=Path(folder);manifest=json.loads((folder/'year_manifest.json').read_text())
    actual={str(p.relative_to(folder)) for p in folder.rglob('*') if p.is_file() and p.name!='year_manifest.json'}
    if actual!=set(manifest['files']):raise ValueError('Completed year file inventory changed')
    for name,r in manifest['files'].items():
        raw=(folder/name).read_bytes()
        if len(raw)!=r['bytes'] or digest(raw)!=r['sha256']:raise ValueError(f'Completed-year checksum changed: {name}')
    return manifest


In [69]:
CONTRACT={'run_id':RUN_ID,'policy':HISTORY_POLICY,'target_years':HISTORY_YEARS,'bbox':list(BBOX),
          'boundary_sha256':BOUNDARY_SHA256,'helper_code_sha256':HELPER_CODE_SHA256,'benchmark_reference':BENCHMARK_REFERENCE,'audit_mirror':True}
contract_path=OUT/'run_contract.json'
if contract_path.exists():assert json.loads(contract_path.read_text())==CONTRACT,'Run contract changed; preserve this run and start a new one.'
else:contract_path.write_text(json.dumps(CONTRACT,indent=2))
(OUT/'region.geojson').write_bytes(BOUNDARY_BYTES)
assert max(HISTORY_YEARS)==2021 and not set(HISTORY_YEARS)&{2022,2023,2024}
print('History contract and reference polygon verified.')


History contract and reference polygon verified.


## 3. Private MAP_KEY and safe cached requests
The key is runtime-only. Every cached file needs an identical request contract and matching checksum. No credential-bearing URLs or provider errors are printed.


In [70]:
MAP_KEY=getpass('NASA FIRMS MAP_KEY (hidden): ').strip()
assert MAP_KEY and MAP_KEY.replace('-','').isalnum(),'Enter a FIRMS MAP_KEY, not an Earthdata password.'
def fetch_bytes(url,private=False):
    status='network/timeout'
    for attempt in range(5):
        try:
            with urlopen(Request(url,headers={'User-Agent':'CinderTrace-history-checkpoint/1.0','X-Requested-With':'XMLHttpRequest'}),timeout=90) as response:payload=response.read()
            if private and MAP_KEY.encode() in payload:raise ValueError('Credential echoed by provider; response was not saved.')
            return payload
        except HTTPError as error:
            status='HTTP '+str(error.code)
            if error.code not in [429,500,502,503,504]:break
        except (URLError,TimeoutError,OSError):status='network/timeout'
        if attempt<4:time.sleep(min(2**(attempt+1),30))
    raise RuntimeError(f'Data request failed ({status}); no zero was written. Wait for service/quota recovery, then resume. Verified files and completed year ZIPs remain cached.') from None

def cached_bytes(path,url,request,private=False):
    path=Path(path);receipt_path=Path(str(path)+'.receipt.json')
    if path.exists() and receipt_path.exists():
        raw=path.read_bytes();r=json.loads(receipt_path.read_text())
        assert r['request']==request and r['bytes']==len(raw) and r['sha256']==digest(raw),'Cached input changed; preserve it and start a fresh run for review.'
        return raw,r
    raw=fetch_bytes(url,private=private)
    r={'request':request,'retrieved_utc':datetime.now(timezone.utc).isoformat(),'bytes':len(raw),'sha256':digest(raw)}
    temporary=Path(str(path)+'.part');temporary.write_bytes(raw);temporary.replace(path)
    receipt_path.write_text(json.dumps(r,indent=2))
    return raw,r
print('Private key entered; cached request helpers ready.')


Private key entered; cached request helpers ready.


## 4. Pin availability and public outage snapshots
These snapshots stay fixed for this run. A new snapshot needs a new run. Historical notice rows that cannot be parsed are retained explicitly; they are not silently interpreted as an absence of outages. Neither the parsed inventory nor archive date metadata certifies complete local coverage.


In [71]:
snap=OUT/'snapshots'
raw,AVAILABILITY_RECEIPT=cached_bytes(snap/'nasa_availability.csv',
    'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/'+MAP_KEY+'/all',{'service':'NASA FIRMS data_availability','source':'all'},private=True)
AVAILABLE=availability_table(raw)
PLANS,WINDOWS=history_plan(AVAILABLE,HISTORY_YEARS)
PLANS.to_parquet(OUT/'official_history_plan.parquet',index=False)
PUBLIC={};PUBLIC_RECEIPTS={}
for label,url in PUBLIC_SOURCES.items():
    name=label+('.json' if label=='missing' else '.html')
    PUBLIC[label],PUBLIC_RECEIPTS[label]=cached_bytes(snap/name,url,{'public_source':url})
GAPS,MISSING_DATES,UNPARSED_EVIDENCE=history_evidence(PUBLIC,HISTORY_YEARS)
GAPS.to_parquet(OUT/'documented_global_gaps.parquet',index=False)
MISSING_DATES.to_parquet(OUT/'upstream_missing_standard_dates.parquet',index=False)
UNPARSED_EVIDENCE.to_parquet(OUT/'unparsed_outage_evidence.parquet',index=False)
print('Target years:',HISTORY_YEARS[0],'through',HISTORY_YEARS[-1],'| planned NASA requests:',len(WINDOWS))
print('Parsed global intervals:',len(GAPS),'| listed missing standard dates:',len(MISSING_DATES),'| unparsed evidence rows:',len(UNPARSED_EVIDENCE))
display(PLANS[PLANS.year.isin([2000,2002,2012,2018,2021])])


Target years: 2000 through 2021 | planned NASA requests: 2555
Parsed global intervals: 750 | listed missing standard dates: 55 | unparsed evidence rows: 0


,year,source,status,start,end,days
0,2000,MODIS_SP,partial_year_advertised,2000-11-01,2001-01-01,61
1,2000,VIIRS_SNPP_SP,outside_advertised_archive_range,None,None,0
2,2000,VIIRS_NOAA20_SP,outside_advertised_archive_range,None,None,0
6,2002,MODIS_SP,full_year_advertised,2002-01-01,2003-01-01,365
7,2002,VIIRS_SNPP_SP,outside_advertised_archive_range,None,None,0
8,2002,VIIRS_NOAA20_SP,outside_advertised_archive_range,None,None,0
36,2012,MODIS_SP,full_year_advertised,2012-01-01,2013-01-01,366
37,2012,VIIRS_SNPP_SP,partial_year_advertised,2012-01-20,2013-01-01,347
38,2012,VIIRS_NOAA20_SP,outside_advertised_archive_range,None,None,0
54,2018,MODIS_SP,full_year_advertised,2018-01-01,2019-01-01,365


## 5. Year processing and immediate evidence packaging
Each year uses NASA standard responses as canonical input. The complete mirror extract and NRT rows remain in the year audit files. A mirror mismatch stays visible and does not cause NRT-to-standard relabelling. Every canonical row has a raw response hash and zero-based parsed CSV row ordinal. Repeated native identities stop for review.


In [72]:
def mirror_fingerprint(url):
    try:
        with urlopen(Request(url,method='HEAD',headers={'User-Agent':'CinderTrace-history-checkpoint/1.0'}),timeout=60) as response:
            r={key:response.headers.get(key) for key in ['ETag','Content-Length','Last-Modified']}
    except Exception:raise RuntimeError('Mirror fingerprint request failed; no snapshot was invented. Resume after resolving access/service.') from None
    assert r['ETag'] and r['Content-Length'],'Incomplete source fingerprint; stop for review.'
    return r

def capture_mirror(year,folder):
    path=folder/'mirror.parquet';receipt_path=folder/'mirror.receipt.json'
    if path.exists() and receipt_path.exists():
        r=json.loads(receipt_path.read_text());raw=path.read_bytes()
        assert r['year']==year and r['boundary_sha256']==BOUNDARY_SHA256 and r['bytes']==len(raw) and r['sha256']==digest(raw)
        f=history_mirror_valid(pd.read_parquet(BytesIO(raw)),year,REGION)
        assert len(f)==r['rows']==r['direct_count'] and r['source_before']==r['source_after']
        return f,r
    url=f'https://data.source.coop/portolan-mirrors/firms-catalog/detections/year={year}/detections.parquet'
    before=mirror_fingerprint(url);con=duckdb.connect()
    try:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=2')
        con.execute('INSTALL httpfs');con.execute('LOAD httpfs');con.execute('INSTALL spatial');con.execute('LOAD spatial')
        schema=con.execute('DESCRIBE SELECT * FROM read_parquet(?)',[url]).df()
        assert str(schema.set_index('column_name').loc['geometry','column_type']).startswith('GEOMETRY'),'Native geometry decoding failed.'
        assert not {'geometry_wkb','audit_lon','audit_lat'}&set(schema.column_name)
        where="acq_date >= CAST(? AS DATE) AND acq_date < CAST(? AS DATE) AND ST_Intersects(geometry,ST_MakeEnvelope(?,?,?,?)) AND ST_Covers(ST_GeomFromText(?),geometry)"
        sql=f'SELECT * EXCLUDE(geometry),ST_AsWKB(geometry) AS geometry_wkb,ST_X(geometry) AS audit_lon,ST_Y(geometry) AS audit_lat FROM read_parquet(?) WHERE {where}'
        params=[url,f'{year}-01-01',f'{year+1}-01-01',*BBOX,REGION.wkt]
        f=con.execute(sql,params).df();f['geometry_wkb']=f.geometry_wkb.map(bytes)
        direct=int(con.execute(f'SELECT count(*) FROM read_parquet(?) WHERE {where}',params).fetchone()[0])
        after=mirror_fingerprint(url)
        assert before==after and len(f)==direct,'Mirror source/count changed; stop and start a new snapshot.'
        history_mirror_valid(f,year,REGION)
    finally:con.close()
    temporary=folder/'mirror.part.parquet';f.to_parquet(temporary,index=False);temporary.replace(path);raw=path.read_bytes()
    r={'year':year,'source_url':url,'source_before':before,'source_after':after,'etag_is_not_checksum':True,
       'sql':sql,'parameters':params,'boundary_sha256':BOUNDARY_SHA256,'rows':len(f),'direct_count':direct,'schema':schema.to_dict('records'),
       'bytes':len(raw),'sha256':digest(raw),'retrieved_utc':datetime.now(timezone.utc).isoformat()}
    receipt_path.write_text(json.dumps(r,indent=2,default=str));return f,r

def package_year(folder):
    m=verify_year_files(folder);year=m['year']
    destination=OUT/f'cindertrace_atlas_lk_history_{year}_{RUN_ID}.zip'
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as archive:
        for name in sorted([*m['files'],'year_manifest.json']):archive.write(folder/name,name)
    receipt={'year':year,'zip_name':destination.name,'bytes':destination.stat().st_size,'sha256':digest(destination.read_bytes()),
             'canonical_standard_rows':m['canonical_standard_rows'],'completed_utc':m['completed_utc']}
    (OUT/f'year_{year}_zip_receipt.json').write_text(json.dumps(receipt,indent=2))
    return receipt

def process_year(year):
    assert year in HISTORY_YEARS and year<2022
    folder=OUT/'years'/str(year);folder.mkdir(exist_ok=True)
    if (folder/'year_manifest.json').exists():
        m=verify_year_files(folder)
        assert m['run_contract_sha256']==digest((OUT/'run_contract.json').read_bytes()),'Completed year belongs to a different contract.'
        return package_year(folder)
    cache=folder/'nasa_cache';cache.mkdir(exist_ok=True)
    mirror,mirror_receipt=capture_mirror(year,folder)
    plan=PLANS[PLANS.year.eq(year)].copy();windows=[w for w in WINDOWS if w[0]==year];parts=[];receipts=[]
    for index,(_,source,start,end) in enumerate(windows,1):
        request={'year':year,'source':source,'start_inclusive_utc':start,'end_exclusive_utc':end,'bbox':list(BBOX),'boundary_sha256':BOUNDARY_SHA256}
        name=f'{source}_{start}_{end}.csv';path=cache/name
        url='https://firms.modaps.eosdis.nasa.gov/api/area/csv/'+MAP_KEY+'/'+source+'/'+','.join(map(str,BBOX))+'/'+str((pd.Timestamp(end)-pd.Timestamp(start)).days)+'/'+start
        payload,receipt=cached_bytes(path,url,request,private=True)
        try:f=parse_official_csv(payload,source,start,end,REGION)
        except ValueError as error:raise ValueError(f'Unverified NASA response for {source} {start} to {end}: {error}. Raw bytes retained; no zero was written.') from None
        f['source_row_ordinal']=f.index.to_numpy();f['source_response_sha256']=receipt['sha256'];f['source_relative_path']='nasa_cache/'+name;f['source_stream']=source
        f['request_start_utc']=start;f['request_end_utc_exclusive']=end
        parts.append(f);receipts.append({**receipt,'regional_rows':len(f),'status':'native_schema_and_region_checked'})
        if index%10==0 or index==len(windows):print(f'{year}: {index}/{len(windows)} NASA windows; {sum(len(p) for p in parts):,} regional standard records.')
        time.sleep(REQUEST_PAUSE_SECONDS)
    canonical=history_canonical(parts)
    if len(canonical):assert pd.to_datetime(canonical.acq_datetime,utc=True).dt.year.eq(year).all()
    covered=pd.Series(False,index=mirror.index);times=pd.to_datetime(mirror.acq_datetime,utc=True);comparisons={}
    for row in plan.itertuples():
        sats=[s for s in HISTORY_SATELLITES if PLATFORM_SOURCE[s]==row.source]
        mask=mirror.quality.eq('sp')&mirror.satellite.isin(sats)
        if row.days:
            mask&=times.ge(pd.Timestamp(row.start,tz='UTC'))&times.lt(pd.Timestamp(row.end,tz='UTC'))
            covered|=mask
            comparisons[row.source]=compare_native(mirror.loc[mask],canonical[canonical.satellite.isin(sats)])
        else:comparisons[row.source]={'status':row.status,'canonical_query_rows':None}
    excluded=mirror.loc[~covered].copy();excluded['not_reconciled_reason']=np.where(excluded.quality.ne('sp'),'NRT_audit_only','outside_advertised_standard_interval')
    excluded.to_parquet(folder/'mirror_not_reconciled.parquet',index=False)
    mirror.loc[mirror.quality.eq('nrt')].to_parquet(folder/'mirror_nrt_audit_only.parquet',index=False)
    daily,monthly,grid=history_grids(year,canonical,plan,GAPS,MISSING_DATES)
    canonical.to_parquet(folder/'observations_standard.parquet',index=False)
    daily.to_parquet(folder/'daily_status.parquet',index=False);monthly.to_parquet(folder/'monthly_status.parquet',index=False);grid.to_parquet(folder/'native_standard_filter_grid.parquet',index=False)
    plan.to_parquet(folder/'official_request_plan.parquet',index=False)
    (folder/'mirror_standard_comparison.json').write_text(json.dumps(comparisons,indent=2,allow_nan=False))
    (folder/'request_receipts.json').write_text(json.dumps(receipts,indent=2))
    (folder/'region.geojson').write_bytes(BOUNDARY_BYTES)
    (folder/'run_contract.json').write_bytes((OUT/'run_contract.json').read_bytes())
    snapshot_dir=folder/'evidence';snapshot_dir.mkdir(exist_ok=True)
    for p in sorted((OUT/'snapshots').iterdir()):
        if p.is_file():shutil.copy2(p,snapshot_dir/p.name)
    for name in ['documented_global_gaps.parquet','upstream_missing_standard_dates.parquet','unparsed_outage_evidence.parquet']:shutil.copy2(OUT/name,snapshot_dir/name)
    conflicts=daily[daily.upstream_lists_missing_standard_data&daily.recorded_detections.fillna(0).gt(0)]
    conflicts.to_parquet(folder/'missing_date_positive_record_review.parquet',index=False)
    m={'year':year,'run_id':RUN_ID,'run_contract_sha256':digest((OUT/'run_contract.json').read_bytes()),'policy':HISTORY_POLICY,
       'availability_receipt':AVAILABILITY_RECEIPT,'public_source_receipts':PUBLIC_RECEIPTS,'mirror_receipt':mirror_receipt,
       'expected_requests':len(windows),'completed_requests':len(receipts),'canonical_standard_rows':len(canonical),
       'mirror_rows':len(mirror),'mirror_nrt_audit_rows':int(mirror.quality.eq('nrt').sum()),'mirror_standard_comparison_status':{s:r['status'] for s,r in comparisons.items()},
       'platform_year_counts':canonical.groupby('satellite').size().to_dict(),'daily_rows':len(daily),'monthly_rows':len(monthly),'native_grid_rows':len(grid),
       'unsupported_platform_days':int(daily.recorded_detections.isna().sum()),'missing_date_positive_records_for_review':len(conflicts),
       'coverage_verified':False,'harmonization_applied':False,'production_export_enabled':False,'frontend_changed':False,
       'packages':{p:metadata.version(p) for p in ['duckdb','pandas','pyarrow','numpy','shapely']},'completed_utc':datetime.now(timezone.utc).isoformat(),'files':{}}
    for path in sorted(folder.rglob('*')):
        if path.is_file() and path.name!='year_manifest.json':
            payload=path.read_bytes();assert MAP_KEY.encode() not in payload,'Credential found in output; stop before bundling.'
            m['files'][str(path.relative_to(folder))]={'bytes':len(payload),'sha256':digest(payload)}
    payload=json.dumps(m,indent=2,default=str,allow_nan=False).encode();assert MAP_KEY.encode() not in payload
    (folder/'year_manifest.json').write_bytes(payload)
    return package_year(folder)
print('Year processing helpers ready. No earlier history has been downloaded yet.')


Year processing helpers ready. No earlier history has been downloaded yet.


## Batch and single-download functions
Each batch skips completed years. The download function verifies saved files and year ZIP hashes before creating one combined archive. It never zips the entire Drive folder or includes incomplete-year files.


In [73]:
"""Operational batching and single-download packaging; does not change data contracts."""
from pathlib import Path
import json,zipfile

def verified_year_zip(out,year,package_year):
    out=Path(out);contract_path=out/'run_contract.json';contract=json.loads(contract_path.read_text());run_id=contract['run_id']
    folder=out/'years'/str(year);m=verify_year_files(folder)
    if m['year']!=year or m['run_contract_sha256']!=digest(contract_path.read_bytes()):raise ValueError('Completed year belongs to a different run contract')
    expected_name=f'cindertrace_atlas_lk_history_{year}_{run_id}.zip';receipt_path=out/f'year_{year}_zip_receipt.json';path=out/expected_name
    if receipt_path.exists() and path.exists():
        r=json.loads(receipt_path.read_text())
        if r['zip_name']!=expected_name or r['year']!=year or r['canonical_standard_rows']!=m['canonical_standard_rows']:raise ValueError('Year ZIP receipt contract differs')
        raw=path.read_bytes()
        if len(raw)!=r['bytes'] or digest(raw)!=r['sha256']:raise ValueError('Saved year ZIP changed; preserve it for review')
        return r
    r=package_year(folder)
    if r['zip_name']!=expected_name:raise ValueError('Packager returned an unexpected year ZIP name')
    return verified_year_zip(out,year,package_year)

def run_fixed_history_batch(out,years,target_years,process_year,package_year):
    years=list(years);target_years=list(target_years)
    if not years or len(years)>3 or len(set(years))!=len(years) or any(y not in target_years for y in years):raise ValueError('Batch must contain up to three distinct target history years')
    receipts=[]
    for year in years:
        folder=Path(out)/'years'/str(year)
        if (folder/'year_manifest.json').is_file():
            r=verified_year_zip(out,year,package_year)
            print(f'{year}: reused completed evidence; no downloads.')
        else:
            process_year(year)
            r=verified_year_zip(out,year,package_year)
            print(f'{year}: completed and saved.')
        receipts.append(r)
    return receipts

def collect_history_bundle(out,target_years,package_year,benchmark_reference,require_all=True):
    """Create one ZIP of verified year ZIPs and evidence index; partial capture stays explicit."""
    out=Path(out);target_years=list(target_years)
    if len(set(target_years))!=len(target_years) or not target_years or any(y<2000 or y>2021 for y in target_years):raise ValueError('Invalid target history years')
    contract_path=out/'run_contract.json';contract=json.loads(contract_path.read_text());run_id=contract['run_id']
    if contract['target_years']!=target_years:raise ValueError('Bundle target differs from the pinned run')
    completed=[y for y in target_years if (out/'years'/str(y)/'year_manifest.json').is_file()]
    remaining=[y for y in target_years if y not in completed]
    if require_all and remaining:raise ValueError(f'History is incomplete; missing years: {remaining}. Run the remaining batch cells, or use require_all=False for an explicitly partial download.')
    if not completed:raise ValueError('No completed year evidence is available to download')
    receipts=[verified_year_zip(out,y,package_year) for y in completed]
    progress={'run_id':run_id,'target_years':target_years,'completed_years':completed,'remaining_years':remaining,
      'capture_status':'all_target_years_captured' if not remaining else 'partial_capture',
      'year_zip_receipts':receipts,'benchmark_reference':benchmark_reference,'run_contract_sha256':digest(contract_path.read_bytes()),
      'coverage_verified':False,'harmonization_applied':False,'frontend_changed':False}
    entries={}
    for r in receipts:entries['year_zips/'+r['zip_name']]=out/r['zip_name']
    shared=['run_contract.json','region.geojson','official_history_plan.parquet','documented_global_gaps.parquet','upstream_missing_standard_dates.parquet','unparsed_outage_evidence.parquet']
    for name in shared:
        if not (out/name).is_file():raise ValueError(f'Missing shared evidence file: {name}')
        entries['shared/'+name]=out/name
    for path in sorted((out/'snapshots').iterdir()):
        if path.is_file():entries['shared/snapshots/'+path.name]=path
    progress_raw=json.dumps(progress,indent=2,allow_nan=False).encode()
    index={**progress,'files':{}}
    for name,path in entries.items():
        raw=path.read_bytes();index['files'][name]={'bytes':len(raw),'sha256':digest(raw)}
    index['files']['progress.json']={'bytes':len(progress_raw),'sha256':digest(progress_raw)}
    label='complete_2000_2021' if not remaining else 'partial_completed_years'
    destination=out/f'cindertrace_atlas_lk_history_{label}_{run_id}.zip';temporary=Path(str(destination)+'.part')
    # Year ZIPs are already compressed. Store them to avoid repeatedly compressing the same data.
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_STORED) as archive:
        for name,path in entries.items():archive.write(path,name)
        archive.writestr('progress.json',progress_raw)
        archive.writestr('bundle_manifest.json',json.dumps(index,indent=2,allow_nan=False))
    temporary.replace(destination)
    (out/'progress.json').write_bytes(progress_raw)
    return {'path':destination,'sha256':digest(destination.read_bytes()),'bytes':destination.stat().st_size,'manifest':index}


In [74]:
def run_history_batch(years):
    return run_fixed_history_batch(OUT,years,HISTORY_YEARS,process_year,package_year)

def download_history_bundle(require_all=True,download=True):
    result=collect_history_bundle(OUT,HISTORY_YEARS,package_year,BENCHMARK_REFERENCE,require_all=require_all)
    print('ONE FILE:',result['path'])
    print('Completed years:',result['manifest']['completed_years'])
    print('Remaining years:',result['manifest']['remaining_years'])
    print('SHA256:',result['sha256'])
    print('Size:',round(result['bytes']/1024**2,2),'MiB')
    if download:
        if IN_COLAB:
            from google.colab import files
            files.download(str(result['path']))
        else:
            from IPython.display import FileLink
            display(FileLink(str(result['path'])))
    return result


## Batch 1: 2000–2002
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [75]:
batch_1_receipts=run_history_batch([2000, 2001, 2002])


2000: reused completed evidence; no downloads.
2001: reused completed evidence; no downloads.
2002: reused completed evidence; no downloads.


## Batch 2: 2003–2005
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [76]:
batch_2_receipts=run_history_batch([2003, 2004, 2005])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2003: 10/73 NASA windows; 7 regional standard records.
2003: 20/73 NASA windows; 52 regional standard records.
2003: 30/73 NASA windows; 120 regional standard records.
2003: 40/73 NASA windows; 341 regional standard records.
2003: 50/73 NASA windows; 962 regional standard records.
2003: 60/73 NASA windows; 1,491 regional standard records.
2003: 70/73 NASA windows; 1,493 regional standard records.
2003: 73/73 NASA windows; 1,505 regional standard records.
2003: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2004: 10/74 NASA windows; 63 regional standard records.
2004: 20/74 NASA windows; 250 regional standard records.
2004: 30/74 NASA windows; 335 regional standard records.
2004: 40/74 NASA windows; 747 regional standard records.
2004: 50/74 NASA windows; 1,789 regional standard records.
2004: 60/74 NASA windows; 1,912 regional standard records.
2004: 70/74 NASA windows; 1,912 regional standard records.
2004: 74/74 NASA windows; 1,915 regional standard records.
2004: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2005: 10/73 NASA windows; 31 regional standard records.
2005: 20/73 NASA windows; 206 regional standard records.
2005: 30/73 NASA windows; 238 regional standard records.
2005: 40/73 NASA windows; 623 regional standard records.
2005: 50/73 NASA windows; 1,411 regional standard records.
2005: 60/73 NASA windows; 1,804 regional standard records.
2005: 70/73 NASA windows; 1,806 regional standard records.
2005: 73/73 NASA windows; 1,808 regional standard records.
2005: completed and saved.


## Batch 3: 2006–2008
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [77]:
batch_3_receipts=run_history_batch([2006, 2007, 2008])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2006: 10/73 NASA windows; 24 regional standard records.
2006: 20/73 NASA windows; 79 regional standard records.
2006: 30/73 NASA windows; 154 regional standard records.
2006: 40/73 NASA windows; 582 regional standard records.
2006: 50/73 NASA windows; 1,184 regional standard records.
2006: 60/73 NASA windows; 1,320 regional standard records.
2006: 70/73 NASA windows; 1,322 regional standard records.
2006: 73/73 NASA windows; 1,324 regional standard records.
2006: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2007: 10/73 NASA windows; 29 regional standard records.
2007: 20/73 NASA windows; 232 regional standard records.
2007: 30/73 NASA windows; 349 regional standard records.
2007: 40/73 NASA windows; 617 regional standard records.
2007: 50/73 NASA windows; 1,001 regional standard records.
2007: 60/73 NASA windows; 1,217 regional standard records.
2007: 70/73 NASA windows; 1,225 regional standard records.
2007: 73/73 NASA windows; 1,226 regional standard records.
2007: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: 10/74 NASA windows; 9 regional standard records.
2008: 20/74 NASA windows; 32 regional standard records.
2008: 30/74 NASA windows; 114 regional standard records.
2008: 40/74 NASA windows; 377 regional standard records.
2008: 50/74 NASA windows; 969 regional standard records.
2008: 60/74 NASA windows; 1,383 regional standard records.
2008: 70/74 NASA windows; 1,400 regional standard records.
2008: 74/74 NASA windows; 1,403 regional standard records.
2008: completed and saved.


## Batch 4: 2009–2011
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [78]:
batch_4_receipts=run_history_batch([2009, 2010, 2011])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009: 10/73 NASA windows; 120 regional standard records.
2009: 20/73 NASA windows; 273 regional standard records.
2009: 30/73 NASA windows; 333 regional standard records.
2009: 40/73 NASA windows; 874 regional standard records.
2009: 50/73 NASA windows; 1,562 regional standard records.
2009: 60/73 NASA windows; 1,901 regional standard records.
2009: 70/73 NASA windows; 1,905 regional standard records.
2009: 73/73 NASA windows; 1,905 regional standard records.
2009: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2010: 10/73 NASA windows; 20 regional standard records.
2010: 20/73 NASA windows; 289 regional standard records.
2010: 30/73 NASA windows; 344 regional standard records.
2010: 40/73 NASA windows; 445 regional standard records.
2010: 50/73 NASA windows; 900 regional standard records.
2010: 60/73 NASA windows; 1,095 regional standard records.
2010: 70/73 NASA windows; 1,096 regional standard records.
2010: 73/73 NASA windows; 1,097 regional standard records.
2010: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2011: 10/73 NASA windows; 3 regional standard records.
2011: 20/73 NASA windows; 46 regional standard records.
2011: 30/73 NASA windows; 68 regional standard records.
2011: 40/73 NASA windows; 382 regional standard records.
2011: 50/73 NASA windows; 867 regional standard records.
2011: 60/73 NASA windows; 1,376 regional standard records.
2011: 70/73 NASA windows; 1,379 regional standard records.
2011: 73/73 NASA windows; 1,379 regional standard records.
2011: completed and saved.


## Batch 5: 2012–2014
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [79]:
batch_5_receipts=run_history_batch([2012, 2013, 2014])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2012: 10/144 NASA windows; 29 regional standard records.
2012: 20/144 NASA windows; 124 regional standard records.
2012: 30/144 NASA windows; 191 regional standard records.
2012: 40/144 NASA windows; 641 regional standard records.
2012: 50/144 NASA windows; 1,609 regional standard records.
2012: 60/144 NASA windows; 1,833 regional standard records.
2012: 70/144 NASA windows; 1,833 regional standard records.
2012: 80/144 NASA windows; 1,931 regional standard records.
2012: 90/144 NASA windows; 2,931 regional standard records.
2012: 100/144 NASA windows; 3,307 regional standard records.
2012: 110/144 NASA windows; 5,427 regional standard records.
2012: 120/144 NASA windows; 10,072 regional standard records.
2012: 130/144 NASA windows; 13,199 regional standard records.
2012: 140/144 NASA windows; 13,235 regional standard records.
2012: 144/144 NASA windows; 13,237 regional standard records.


/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2012: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (5 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2014: 10/146 NASA windows; 41 regional standard records.
2014: 20/146 NASA windows; 255 regional standard records.
2014: 30/146 NASA windows; 319 regional standard records.
2014: 40/146 NASA windows; 562 regional standard records.
2014: 50/146 NASA windows; 1,155 regional standard records.
2014: 60/146 NASA windows; 1,382 regional standard records.
2014: 70/146 NASA windows; 1,390 regional standard records.
2014: 80/146 NASA windows; 1,507 regional standard records.
2014: 90/146 NASA windows; 3,023 regional standard records.
2014: 100/146 NASA windows; 4,216 regional standard records.
2014: 110/146 NASA windows; 5,292 regional standard records.
2014: 120/146 NASA windows; 8,773 regional standard records.
2014: 130/146 NASA windows; 11,579 regional standard records.
2014: 140/146 NASA windows; 11,668 regional standard records.
2014: 146/146 NASA windows; 11,673 regional standard records.


/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2014: completed and saved.


## Batch 6: 2015–2017
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [80]:
batch_6_receipts=run_history_batch([2015, 2016, 2017])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2015: 10/146 NASA windows; 17 regional standard records.
2015: 20/146 NASA windows; 60 regional standard records.
2015: 30/146 NASA windows; 79 regional standard records.
2015: 40/146 NASA windows; 176 regional standard records.
2015: 50/146 NASA windows; 548 regional standard records.
2015: 60/146 NASA windows; 694 regional standard records.
2015: 70/146 NASA windows; 694 regional standard records.
2015: 80/146 NASA windows; 753 regional standard records.
2015: 90/146 NASA windows; 1,182 regional standard records.
2015: 100/146 NASA windows; 1,331 regional standard records.
2015: 110/146 NASA windows; 1,584 regional standard records.
2015: 120/146 NASA windows; 3,604 regional standard records.
2015: 130/146 NASA windows; 5,014 regional standard records.
2015: 140/146 NASA windows; 5,057 regional standard records.
2015: 146/146 NASA windows; 5,091 regional standard records.


/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2015: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (5 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017: 10/146 NASA windows; 37 regional standard records.
2017: 20/146 NASA windows; 89 regional standard records.
2017: 30/146 NASA windows; 159 regional standard records.
2017: 40/146 NASA windows; 373 regional standard records.
2017: 50/146 NASA windows; 635 regional standard records.
2017: 60/146 NASA windows; 739 regional standard records.
2017: 70/146 NASA windows; 744 regional standard records.
2017: 80/146 NASA windows; 956 regional standard records.
2017: 90/146 NASA windows; 1,379 regional standard records.
2017: 100/146 NASA windows; 2,083 regional standard records.
2017: 110/146 NASA windows; 3,082 regional standard records.
2017: 120/146 NASA windows; 5,194 regional standard records.
2017: 130/146 NASA windows; 6,328 regional standard records.
2017: 140/146 NASA windows; 6,401 regional standard records.
2017: 146/146 NASA windows; 6,428 regional standard records.


/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2017: completed and saved.


## Batch 7: 2018–2020
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [81]:
batch_7_receipts=run_history_batch([2018, 2019, 2020])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2018: 10/201 NASA windows; 48 regional standard records.
2018: 20/201 NASA windows; 141 regional standard records.
2018: 30/201 NASA windows; 172 regional standard records.
2018: 40/201 NASA windows; 297 regional standard records.
2018: 50/201 NASA windows; 770 regional standard records.
2018: 60/201 NASA windows; 942 regional standard records.
2018: 70/201 NASA windows; 943 regional standard records.
2018: 80/201 NASA windows; 1,107 regional standard records.
2018: 90/201 NASA windows; 2,022 regional standard records.
2018: 100/201 NASA windows; 2,458 regional standard records.
2018: 110/201 NASA windows; 2,677 regional standard records.
2018: 120/201 NASA windows; 4,974 regional standard records.
2018: 130/201 NASA windows; 8,071 regional standard records.
2018: 140/201 NASA windows; 8,089 regional standard records.
2018: 150/201 NASA windows; 8,271 regional standard records.
2018: 160/201 NASA windows; 8,537 regional standard records.
2018: 170/201 NASA windows; 9,598 regional stand

/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2018: completed and saved.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (5 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2020: 10/222 NASA windows; 66 regional standard records.
2020: 20/222 NASA windows; 246 regional standard records.
2020: 30/222 NASA windows; 333 regional standard records.
2020: 40/222 NASA windows; 396 regional standard records.
2020: 50/222 NASA windows; 572 regional standard records.
2020: 60/222 NASA windows; 737 regional standard records.
2020: 70/222 NASA windows; 749 regional standard records.
2020: 80/222 NASA windows; 864 regional standard records.
2020: 90/222 NASA windows; 2,515 regional standard records.
2020: 100/222 NASA windows; 4,128 regional standard records.
2020: 110/222 NASA windows; 4,342 regional standard records.
2020: 120/222 NASA windows; 5,040 regional standard records.
2020: 130/222 NASA windows; 7,654 regional standard records.
2020: 140/222 NASA windows; 8,478 regional standard records.
2020: 150/222 NASA windows; 8,522 regional standard records.
2020: 160/222 NASA windows; 9,334 regional standard records.
2020: 170/222 NASA windows; 11,382 regional standa

/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2020: completed and saved.


## Batch 8: 2021
Run this cell in order. Completed years reuse their verified files and make no data requests.


In [82]:
batch_8_receipts=run_history_batch([2021])


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2021: 10/219 NASA windows; 22 regional standard records.
2021: 20/219 NASA windows; 105 regional standard records.
2021: 30/219 NASA windows; 124 regional standard records.
2021: 40/219 NASA windows; 190 regional standard records.
2021: 50/219 NASA windows; 409 regional standard records.
2021: 60/219 NASA windows; 589 regional standard records.
2021: 70/219 NASA windows; 591 regional standard records.
2021: 80/219 NASA windows; 632 regional standard records.
2021: 90/219 NASA windows; 1,329 regional standard records.
2021: 100/219 NASA windows; 1,562 regional standard records.
2021: 110/219 NASA windows; 1,872 regional standard records.
2021: 120/219 NASA windows; 2,681 regional standard records.
2021: 130/219 NASA windows; 4,401 regional standard records.
2021: 140/219 NASA windows; 4,509 regional standard records.
2021: 150/219 NASA windows; 4,568 regional standard records.
2021: 160/219 NASA windows; 5,196 regional standard records.
2021: 170/219 NASA windows; 5,657 regional standar

/tmp/ipykernel_476/4044137064.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(nonempty,ignore_index=True).reindex(columns=columns) if nonempty else parts[0].iloc[:0].reindex(columns=columns).copy()


2021: completed and saved.


## Download once
This cell requires all 2000–2021 year captures to be complete. Save the executed notebook and return this single combined ZIP for independent verification. Observing coverage and cross-sensor calibration remain unverified.


In [83]:
FINAL_BUNDLE=download_history_bundle()


ONE FILE: /content/drive/MyDrive/cindertrace_atlas/rebuild/lk_history_2000_2021/20261114T211825Z_97d5550a/cindertrace_atlas_lk_history_complete_2000_2021_20261114T211825Z_97d5550a.zip
Completed years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
Remaining years: []
SHA256: b51ededac8925967d9d7edc58ea07524df7e72c41e0c02dea8ce5af0fded98c7
Size: 24.72 MiB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>